In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import IntegratedGradients, LayerGradCam, LayerAttribution
from captum.attr import visualization as viz

ig_explainer = IntegratedGradients(model)
cam_explainer = LayerGradCam(model, model.layer4[-1]) 

def run_explainers(img_tensor, raw_img, real_label, pred_label, text_caption):
    zero_base = torch.zeros_like(img_tensor) 
    ig_values, delta = ig_explainer.attribute(img_tensor, zero_base, target=pred_label, return_convergence_delta=True)
    
    cam_values = cam_explainer.attribute(img_tensor, target=pred_label)
    cam_scaled = LayerAttribution.interpolate(cam_values, img_tensor.shape[2:])
    
    ig_np = np.transpose(ig_values.squeeze().cpu().detach().numpy(), (1, 2, 0))
    cam_np = np.transpose(cam_scaled.squeeze().cpu().detach().numpy(), (1, 2, 0))
    img_np = np.transpose(raw_img.squeeze().cpu().numpy(), (1, 2, 0))
    
    f, ax = plt.subplots(1, 3, figsize=(15, 5))
    f.suptitle(f"True: {real_label} | Predicted: {pred_label}", fontsize=16)
    
    viz.visualize_image_attr(ig_np, img_np, method="blended_heat_map",
                             sign="all", show_colorbar=True, title="Integrated Gradients", 
                             plt_fig_axis=(f, ax[0]), use_pyplot=False)
    
    viz.visualize_image_attr(cam_np, img_np, method="blended_heat_map",
                             sign="positive", show_colorbar=True, title="Grad-CAM", 
                             plt_fig_axis=(f, ax[1]), use_pyplot=False)
    
    ax[2].imshow(img_np)
    ax[2].set_title("Base Input")
    ax[2].axis('off')
    
    plt.figtext(0.5, 0.01, text_caption, wrap=True, horizontalalignment='center', fontsize=12)
    plt.show()

good_ones = []
bad_ones = []
labels_map = {0: "Cat", 1: "Dog"}
scratch_pad = 0 

for imgs, lbls in val_loader:
    model_out = model(imgs)
    _, predictions = torch.max(model_out, 1)
    
    for i in range(len(predictions)):
        tnsr = imgs[i].unsqueeze(0)  
        src = imgs[i]                   
        actual = labels_map[lbls[i].item()]
        guess = labels_map[predictions[i].item()]
        
        if predictions[i] == lbls[i] and len(good_ones) < 3:
            good_ones.append((tnsr, src, actual, guess))
        elif predictions[i] != lbls[i] and len(bad_ones) < 1:
            bad_ones.append((tnsr, src, actual, guess))
            
    if len(good_ones) == 3 and len(bad_ones) == 1:
        break

run_explainers(*good_ones[0], "Integrated Gradients highlights the sharp pixel edges. Grad-CAM agrees but is broader.")
run_explainers(*good_ones[1], "High agreement. Integrated Gradients highlights the triangular geometry of the ears.")
run_explainers(*good_ones[2], "Disagreement. Grad-CAM finds the body; IG highlights noisy points on the back.")
run_explainers(*bad_ones[0], "Dataset artifact. Both methods point to the background instead of the subject.")

### Explainer Algorithms

**Integrated Gradients**
**Intuition:** It calculates how the prediction changes as you "fade in" the image from a blank baseline. By accumulating these small gradients, it assigns an importance score to every pixel.
**Role of the Baseline:** Starting from an all-black image forces the model to ignore anything that isn't actively contributing to the class, effectively filtering out "zero-information" areas.
**Limitation:** It’s pretty computationally heavy—since you have to compute gradients for many steps between the blank and the real image, it can be quite slow on larger datasets.

**Grad-CAM**
**Intuition:** It looks at the last convolutional layer to see which feature maps are firing the most for a specific class, then maps that "heat" back onto the original image.
**Role of the Target Layer:** We use the final convolutional layer because it’s the sweet spot; it has enough spatial awareness to know *where* something is, but enough depth to know *what* that thing actually represents.
**Limitation:** It’s a bit blurry. Because it relies on deep feature maps, the heatmaps don't have super high resolution, so it’s hard to pinpoint exact, pixel-perfect boundaries.

### Depth of Interpretation

**Correctly Classified Image 1:**
Both methods actually lined up really well here. Integrated Gradients focused on the sharp lines of the face, while Grad-CAM gave us a general "hot spot" over the same area. It’s a good example of the two methods complementing each other.

**Correctly Classified Image 2:**
Pretty high agreement. Grad-CAM covered the top half of the head, and looking at the Integrated Gradients output, you can see it really narrowed in on the ears. It seems the model is picking up on those specific triangular geometric features to confirm it's a cat.

**Correctly Classified Image 3:**
This one was interesting because they didn't quite match. Grad-CAM was happy just highlighting the whole torso, but Integrated Gradients was all over the place with noisy points. It looks like the model is relying on the texture of the fur rather than just the shape of the dog, which explains the noisy IG output.

**Misclassified Image:**
This was the most telling one. The model completely failed to see the animal. Grad-CAM lit up a random object in the background, and the IG heatmap confirmed it by putting almost all the "weight" on that background object instead of the cat. It's clear the model got "distracted" by a dataset artifact, which definitely points to the need for better training data or more diverse augmentation.